# Fine-tuning LogBERT on Web Logs with LoRA

Этот ноутбук встроен в текущую архитектуру репозитория `LogSight` и переиспользует существующие модули `training/bert_pytorch` + `parsing/dataset` для дообучения уже обученной модели LogBERT из `weights/` с помощью LoRA.

## 1) Paths and Hyperparameters

In [1]:
from pathlib import Path
import json

def find_repo_root(start: Path) -> Path:
    markers = ['weights', 'training', 'parsing', 'clients']
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate LogSight repo root from current working directory')

repo_root = find_repo_root(Path.cwd())
weights_dir = repo_root / 'weights'
data_dir = repo_root / 'logs'
output_dir = repo_root / 'output' / 'web_lora_finetune'
output_dir.mkdir(parents=True, exist_ok=True)

PATHS = {
    'repo_root': str(repo_root),
    'weights_dir': str(weights_dir),
    'data_dir': str(data_dir),
    'output_dir': str(output_dir),
    # Можно заменить на другой structured CSV, если нужно.
    'structured_csv': str(repo_root / 'output' / 'dvwa_vulhab' / 'DVWA_Vulhab.log_structured.csv'),
    # Если structured_csv не существует, будет вызван prepare_dataset для target ниже.
    'prepare_target': 'vulhab',
    'base_checkpoint': str(weights_dir / 'best_bert.pth'),
    'base_vocab': str(weights_dir / 'vocab.pkl'),
}

HYPERPARAMS = {
    'seq_len': 512,
    'max_len': 512,
    'window_size': 20,
    'adaptive_window': True,
    'train_ratio': 1.0,
    'valid_ratio': 0.1,
    'mask_ratio': 0.15,
    'min_len': 2,
    'batch_size': 64,
    'num_workers': 0,
    'epochs': 6,
    'lr': 2e-4,
    'weight_decay': 0.0,
    'grad_accum_steps': 1,
    'warmup_ratio': 0.1,
    'max_grad_norm': 1.0,
    'use_amp': True,
    # Для реконструкции модели, если checkpoint окажется state_dict-only
    'fallback_attn_heads': 4,
}

DATA_PREP = {
    # Обычно для self-supervised pretrain используют нормальные (Label=0) сессии
    'use_labels': [0],
    'session_strategy': 'minute',  # 'minute' | 'fixed'
    'fallback_session_size': 50,
    'train_file_name': 'train',
}

LORA_CONFIG = {
    'r': 8,
    'alpha': 16,
    'dropout': 0.05,
    'target_keywords': [
        'attention.linear_layers',
        'attention.output_linear',
        'feed_forward.w_1',
        'feed_forward.w_2',
    ],
    'freeze_base_model': True,
    # Для веб-логов обычно много новых EventId.
    'allow_vocab_expansion': True,
    # Если vocab расширен, обновляем только новые строки embedding/head.
    'train_new_vocab_rows': True,
}

print(json.dumps({'PATHS': PATHS, 'HYPERPARAMS': HYPERPARAMS, 'DATA_PREP': DATA_PREP, 'LORA_CONFIG': LORA_CONFIG}, indent=2))

{
  "PATHS": {
    "repo_root": "/Users/investing/Documents/LogSight",
    "weights_dir": "/Users/investing/Documents/LogSight/weights",
    "data_dir": "/Users/investing/Documents/LogSight/logs",
    "output_dir": "/Users/investing/Documents/LogSight/output/web_lora_finetune",
    "structured_csv": "/Users/investing/Documents/LogSight/output/dvwa_vulhab/DVWA_Vulhab.log_structured.csv",
    "prepare_target": "vulhab",
    "base_checkpoint": "/Users/investing/Documents/LogSight/weights/best_bert.pth",
    "base_vocab": "/Users/investing/Documents/LogSight/weights/vocab.pkl"
  },
  "HYPERPARAMS": {
    "seq_len": 512,
    "max_len": 512,
    "window_size": 20,
    "adaptive_window": true,
    "train_ratio": 1.0,
    "valid_ratio": 0.1,
    "mask_ratio": 0.15,
    "min_len": 2,
    "batch_size": 64,
    "num_workers": 0,
    "epochs": 6,
    "lr": 0.0002,
    "weight_decay": 0.0,
    "grad_accum_steps": 1,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "use_amp": true,
    "fallba

## 2) Environment Init

In [2]:
import os
import sys
import random
import numpy as np
import torch

os.environ.setdefault('MPLCONFIGDIR', str(output_dir / '.mpl_cache'))
os.environ.setdefault('XDG_CACHE_HOME', str(output_dir / '.xdg_cache'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Device:', device)

Python: 3.14.2 (main, Dec  5 2025, 16:49:16) [Clang 17.0.0 (clang-1700.6.3.2)]
PyTorch: 2.10.0
Device: cpu


## 3) Repository Inspection (Required)

In [3]:
from pathlib import Path

key_files = [
    'training/bert_pytorch/train_log.py',
    'training/bert_pytorch/trainer/pretrain.py',
    'training/bert_pytorch/model/bert.py',
    'training/bert_pytorch/model/log_model.py',
    'training/bert_pytorch/model/attention/multi_head.py',
    'training/bert_pytorch/dataset/log_dataset.py',
    'training/bert_pytorch/dataset/sample.py',
    'training/bert_pytorch/predict_log.py',
    'training/inference/utils.py',
    'parsing/dataset/pipeline.py',
    'parsing/dataset/builders.py',
    'parsing/dataset/loaders.py',
    'parsing/dataset/template.py',
    'weights/best_bert.pth',
    'weights/vocab.pkl',
]

print('Repository root:', repo_root)
for rel in key_files:
    p = repo_root / rel
    status = 'OK' if p.exists() else 'MISSING'
    print(f'[{status}] {rel}')

Repository root: /Users/investing/Documents/LogSight
[OK] training/bert_pytorch/train_log.py
[OK] training/bert_pytorch/trainer/pretrain.py
[OK] training/bert_pytorch/model/bert.py
[OK] training/bert_pytorch/model/log_model.py
[OK] training/bert_pytorch/model/attention/multi_head.py
[OK] training/bert_pytorch/dataset/log_dataset.py
[OK] training/bert_pytorch/dataset/sample.py
[OK] training/bert_pytorch/predict_log.py
[OK] training/inference/utils.py
[OK] parsing/dataset/pipeline.py
[OK] parsing/dataset/builders.py
[OK] parsing/dataset/loaders.py
[OK] parsing/dataset/template.py
[OK] weights/best_bert.pth
[OK] weights/vocab.pkl


In [4]:
import inspect

from training.bert_pytorch.train_log import Trainer as LegacyTrainEntry
from training.bert_pytorch.trainer.pretrain import BERTTrainer
from training.bert_pytorch.predict_log import Predictor as LegacyPredictor
from training.bert_pytorch.dataset.log_dataset import LogDataset
from training.bert_pytorch.dataset.sample import generate_train_valid
from training.inference.utils import load_logbert_checkpoint

print('Train entrypoint:', inspect.getsourcefile(LegacyTrainEntry))
print('Trainer class:', inspect.getsourcefile(BERTTrainer))
print('Predictor (inference):', inspect.getsourcefile(LegacyPredictor))
print('LogDataset:', inspect.getsourcefile(LogDataset))
print('generate_train_valid:', inspect.getsourcefile(generate_train_valid))
print('Checkpoint loader helper:', inspect.getsourcefile(load_logbert_checkpoint))

Matplotlib is building the font cache; this may take a moment.


Train entrypoint: /Users/investing/Documents/LogSight/training/bert_pytorch/train_log.py
Trainer class: /Users/investing/Documents/LogSight/training/bert_pytorch/trainer/pretrain.py
Predictor (inference): /Users/investing/Documents/LogSight/training/bert_pytorch/predict_log.py
LogDataset: /Users/investing/Documents/LogSight/training/bert_pytorch/dataset/log_dataset.py
generate_train_valid: /Users/investing/Documents/LogSight/training/bert_pytorch/dataset/sample.py
Checkpoint loader helper: /Users/investing/Documents/LogSight/training/inference/utils.py


### Почему выбран такой pipeline
В репозитории есть готовые `model`, `dataset`, `collate_fn`, `generate_train_valid` и парсинг веб-логов в `parsing/dataset`.

Существующий `BERTTrainer` оптимизирует *все* параметры модели и не содержит встроенного LoRA/resume/AMP-обвязки в нужном для PEFT виде, поэтому ниже используется кастомный train-loop, но на тех же классах модели/датасета и с тем же MLM-loss (`NLLLoss(ignore_index=0)`).

## 4) Prepare Web Logs Dataset (Structured -> Train Sessions)

In [5]:
import pandas as pd
from parsing.dataset.pipeline import prepare_dataset

structured_csv = Path(PATHS['structured_csv'])
if not structured_csv.exists():
    stats = prepare_dataset(
        target=PATHS['prepare_target'],
        logs_dir=Path(PATHS['data_dir']),
        output_dir=repo_root / 'output',
    )
    structured_csv = Path(stats['structured_path'])
    print('Structured dataset created:', stats)

df_structured = pd.read_csv(structured_csv)
print('Structured CSV:', structured_csv)
print('Rows:', len(df_structured))
print('Columns:', list(df_structured.columns))
if 'Label' in df_structured.columns:
    print('Label distribution:')
    print(df_structured['Label'].value_counts(dropna=False).head(20))

display(df_structured.head(3))

Structured CSV: /Users/investing/Documents/LogSight/output/dvwa_vulhab/DVWA_Vulhab.log_structured.csv
Rows: 110000
Columns: ['LineId', 'Label', 'Id', 'Date', 'Admin', 'Month', 'Day', 'Time', 'AdminAddr', 'Content', 'EventId', 'EventTemplate']
Label distribution:
Label
0    100000
1     10000
Name: count, dtype: int64


,LineId,Label,Id,Date,Admin,Month,Day,Time,AdminAddr,Content,EventId,EventTemplate
0,1,0,458086c01f84d5da4f08b341826d5be4,2026-03-16,-,3,16,14:04:33.808,-,"{""ts"":""2026-03-16T13:59:50+00:00"",""level"":""INF...",34f3e735,"{""ts"":""<*>"",""level"":""INFO"",""action"":""request"",..."
1,2,0,31100b06dd3f4eb7d691c2e530b90848,2026-03-16,-,3,16,14:04:33.808,-,"{""ts"":""2026-03-16T13:59:50+00:00"",""level"":""INF...",6643f780,"{""ts"":""<*>"",""level"":""INFO"",""action"":""response""..."
2,3,0,b7dd0e911b59be124a5738da83cf60de,2026-03-16,-,3,16,14:04:33.808,-,"{""ts"":""2026-03-16T13:59:50+00:00"",""level"":""INF...",34f3e735,"{""ts"":""<*>"",""level"":""INFO"",""action"":""request"",..."


In [ ]:
import numpy as np

def build_train_sessions_from_structured(
    df: pd.DataFrame,
    use_labels=None,
    session_strategy: str = 'minute',
    fallback_session_size: int = 50,
    min_session_len: int = 2,
):
    work = df.copy()

    if use_labels is not None and 'Label' in work.columns:
        allowed = {str(x) for x in use_labels}
        work = work[work['Label'].astype(str).isin(allowed)].copy()

    if 'LineId' in work.columns:
        work['__line_id_num'] = pd.to_numeric(work['LineId'], errors='coerce').fillna(0).astype(int)
    else:
        work['__line_id_num'] = np.arange(1, len(work) + 1)

    date_part = work.get('Date', pd.Series(['-'] * len(work))).astype(str)
    time_part = work.get('Time', pd.Series(['-'] * len(work))).astype(str)
    dt_series = pd.to_datetime((date_part + ' ' + time_part).str.strip(), errors='coerce', utc=True)
    work['__dt'] = dt_series

    work = work.sort_values(['__dt', '__line_id_num'], na_position='last').reset_index(drop=True)

    if session_strategy == 'minute':
        minute_key = work['__dt'].dt.strftime('%Y-%m-%d %H:%M')
        fallback = (np.arange(len(work)) // max(1, fallback_session_size)).astype(str)
        work['__session'] = np.where(minute_key.notna(), minute_key, 'fallback_' + fallback)
    elif session_strategy == 'fixed':
        work['__session'] = (np.arange(len(work)) // max(1, fallback_session_size)).astype(str)
    else:
        raise ValueError(f'Unknown session_strategy={session_strategy}')

    lines = []
    meta_rows = []

    for session_id, grp in work.groupby('__session', sort=False):
        event_ids = grp['EventId'].astype(str).tolist()
        if len(event_ids) < min_session_len:
            continue

        times = grp['__dt'].tolist()
        deltas = [0.0]
        for i in range(1, len(times)):
            prev_t, cur_t = times[i - 1], times[i]
            if pd.notna(prev_t) and pd.notna(cur_t):
                delta = max(0.0, float((cur_t - prev_t).total_seconds()))
            else:
                delta = 0.0
            deltas.append(delta)

        tokens = [f'{eid},{delta:.6f}' for eid, delta in zip(event_ids, deltas)]
        lines.append(' '.join(tokens))

        label_value = grp['Label'].mode().iloc[0] if 'Label' in grp.columns and not grp['Label'].isna().all() else 0
        meta_rows.append({
            'session_id': session_id,
            'num_events': len(event_ids),
            'label_mode': label_value,
        })

    return lines, pd.DataFrame(meta_rows)

session_lines, session_meta = build_train_sessions_from_structured(
    df_structured,
    use_labels=DATA_PREP['use_labels'],
    session_strategy=DATA_PREP['session_strategy'],
    fallback_session_size=DATA_PREP['fallback_session_size'],
    min_session_len=HYPERPARAMS['min_len'],
)

train_data_dir = output_dir / 'data'
train_data_dir.mkdir(parents=True, exist_ok=True)
train_file_path = train_data_dir / DATA_PREP['train_file_name']
with train_file_path.open('w', encoding='utf-8') as f:
    for line in session_lines:
        f.write(line + '\n')

session_meta_path = train_data_dir / 'session_meta.csv'
session_meta.to_csv(session_meta_path, index=False)

print('Train file:', train_file_path)
print('Num sessions:', len(session_lines))
print('Session meta:', session_meta_path)
display(session_meta.head(5))

## 5) Load/Extend Vocab

In [ ]:
import pickle
from training.bert_pytorch.dataset.vocab import WordVocab

def collect_event_ids_from_train_file(path: Path):
    out = set()
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            for token in line.split():
                event_id = token.split(',', 1)[0]
                out.add(event_id)
    return out

vocab = WordVocab.load_vocab(PATHS['base_vocab'])
original_vocab_size = len(vocab)

web_event_ids = collect_event_ids_from_train_file(train_file_path)
missing_tokens = sorted([tok for tok in web_event_ids if tok not in vocab.stoi])

print('Base vocab size:', original_vocab_size)
print('Web event ids:', len(web_event_ids))
print('Missing in base vocab:', len(missing_tokens))

if LORA_CONFIG['allow_vocab_expansion'] and missing_tokens:
    for token in missing_tokens:
        vocab.itos.append(token)
        vocab.stoi[token] = len(vocab.itos) - 1

    vocab_artifacts_dir = output_dir / 'artifacts'
    vocab_artifacts_dir.mkdir(parents=True, exist_ok=True)
    vocab_path_for_training = vocab_artifacts_dir / 'vocab_web_extended.pkl'
    with vocab_path_for_training.open('wb') as f:
        pickle.dump(vocab, f)
    print('Extended vocab saved:', vocab_path_for_training)
else:
    vocab_path_for_training = Path(PATHS['base_vocab'])

print('Training vocab size:', len(vocab))
print('Training vocab path:', vocab_path_for_training)

## 6) Load Base LogBERT Weights (Required)

In [ ]:
import torch.nn as nn
from training.bert_pytorch.model.bert import BERT
from training.bert_pytorch.model.log_model import BERTLog
from training.inference.utils import safe_torch_load

def _extract_state_dict(state_obj):
    state = state_obj
    if isinstance(state, dict) and any(k in state for k in ('state_dict', 'model_state_dict')):
        state = state.get('state_dict', state.get('model_state_dict'))
    return state

def infer_model_meta_from_state_dict(state_dict, fallback_heads=4):
    token_w = state_dict['bert.embedding.token.weight']
    vocab_size = token_w.shape[0]
    hidden = token_w.shape[1]

    layer_ids = []
    for k in state_dict.keys():
        if k.startswith('bert.transformer_blocks.'):
            parts = k.split('.')
            if len(parts) > 2 and parts[2].isdigit():
                layer_ids.append(int(parts[2]))
    n_layers = (max(layer_ids) + 1) if layer_ids else 4

    is_time = any('time_embed' in k for k in state_dict.keys())

    return {
        'vocab_size': int(vocab_size),
        'hidden': int(hidden),
        'n_layers': int(n_layers),
        'attn_heads': int(fallback_heads),
        'is_time': bool(is_time),
    }

def load_base_logbert(checkpoint_path: Path, device: torch.device):
    raw = safe_torch_load(str(checkpoint_path), device)

    if isinstance(raw, nn.Module):
        model = raw.to(device)
        meta = {
            'vocab_size': int(model.bert.embedding.token.num_embeddings),
            'hidden': int(model.bert.hidden),
            'n_layers': int(model.bert.n_layers),
            'attn_heads': int(model.bert.attn_heads),
            'is_time': bool(getattr(model.bert.embedding, 'is_time', True)),
        }
        source = 'torch checkpoint contains full nn.Module object'
        return model, meta, source

    state = _extract_state_dict(raw)
    if not isinstance(state, dict):
        raise TypeError(f'Unsupported checkpoint format: {type(raw)}')

    meta = infer_model_meta_from_state_dict(state, fallback_heads=HYPERPARAMS['fallback_attn_heads'])
    bert = BERT(
        vocab_size=meta['vocab_size'],
        max_len=HYPERPARAMS['max_len'],
        hidden=meta['hidden'],
        n_layers=meta['n_layers'],
        attn_heads=meta['attn_heads'],
        is_logkey=True,
        is_time=meta['is_time'],
    )
    model = BERTLog(bert=bert, vocab_size=meta['vocab_size'])
    model.load_state_dict(state, strict=False)
    model = model.to(device)
    source = 'torch checkpoint contains state_dict-like object'
    return model, meta, source

base_checkpoint_path = Path(PATHS['base_checkpoint'])
base_model, base_model_meta, base_model_source = load_base_logbert(base_checkpoint_path, device)

print('Model checkpoint path:', base_checkpoint_path)
print('Loaded from:', base_model_source)
print('Model meta:', base_model_meta)
print('Current model class:', type(base_model))

## 7) Candidate LoRA Target Modules (Required)

In [ ]:
linear_modules = [name for name, module in base_model.named_modules() if isinstance(module, nn.Linear)]
candidate_lora_targets = [
    name
    for name in linear_modules
    if any(keyword in name for keyword in LORA_CONFIG['target_keywords'])
]

print('All linear modules:', len(linear_modules))
for n in linear_modules:
    print('  ', n)

print('\nCandidate LoRA targets:', len(candidate_lora_targets))
for n in candidate_lora_targets:
    print('  ', n)

if not candidate_lora_targets:
    raise RuntimeError("No LoRA targets found. Adjust LORA_CONFIG['target_keywords'].")

## 8) Attach LoRA + Trainable Params Report (Required)

In [ ]:
import math

class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError('LoRALinear supports nn.Linear only')
        if r <= 0:
            raise ValueError('r must be > 0')

        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False

        self.r = int(r)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.r

        self.dropout = nn.Dropout(dropout) if dropout and dropout > 0 else nn.Identity()

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        self.lora_A = nn.Parameter(torch.zeros(self.r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, self.r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = (self.dropout(x) @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + self.scaling * lora_out

def _get_parent_module_and_child(model: nn.Module, module_name: str):
    parts = module_name.split('.')
    parent = model
    for part in parts[:-1]:
        if part.isdigit():
            parent = parent[int(part)]
        else:
            parent = getattr(parent, part)
    return parent, parts[-1]

def _set_child_module(parent: nn.Module, child_name: str, new_module: nn.Module):
    if child_name.isdigit():
        parent[int(child_name)] = new_module
    else:
        setattr(parent, child_name, new_module)

def inject_lora(model: nn.Module, target_module_names, r: int, alpha: float, dropout: float):
    replaced = []
    for module_name in target_module_names:
        parent, child_name = _get_parent_module_and_child(model, module_name)
        old_module = parent[int(child_name)] if child_name.isdigit() else getattr(parent, child_name)
        if not isinstance(old_module, nn.Linear):
            continue

        new_module = LoRALinear(old_module, r=r, alpha=alpha, dropout=dropout)
        _set_child_module(parent, child_name, new_module)
        replaced.append(module_name)

    return replaced

def resize_vocab_dependent_layers(model: nn.Module, new_vocab_size: int):
    token_emb = model.bert.embedding.token
    lm_head = model.mask_lm.linear

    old_vocab_size = token_emb.num_embeddings
    if new_vocab_size <= old_vocab_size:
        return old_vocab_size

    emb_device = token_emb.weight.device
    emb_dtype = token_emb.weight.dtype
    pad_idx = token_emb.padding_idx

    new_emb = nn.Embedding(new_vocab_size, token_emb.embedding_dim, padding_idx=pad_idx).to(device=emb_device, dtype=emb_dtype)
    nn.init.normal_(new_emb.weight, mean=0.0, std=0.02)
    with torch.no_grad():
        new_emb.weight[:old_vocab_size].copy_(token_emb.weight)

    head_device = lm_head.weight.device
    head_dtype = lm_head.weight.dtype
    new_head = nn.Linear(lm_head.in_features, new_vocab_size, bias=lm_head.bias is not None).to(device=head_device, dtype=head_dtype)
    nn.init.normal_(new_head.weight, mean=0.0, std=0.02)
    if new_head.bias is not None:
        nn.init.zeros_(new_head.bias)

    with torch.no_grad():
        new_head.weight[:old_vocab_size].copy_(lm_head.weight)
        if lm_head.bias is not None and new_head.bias is not None:
            new_head.bias[:old_vocab_size].copy_(lm_head.bias)

    model.bert.embedding.token = new_emb
    model.mask_lm.linear = new_head

    return old_vocab_size

def enable_new_vocab_row_training(model: nn.Module, old_vocab_size: int):
    handles = []
    token_emb = model.bert.embedding.token
    lm_head = model.mask_lm.linear

    token_emb.weight.requires_grad_(True)
    lm_head.weight.requires_grad_(True)
    if lm_head.bias is not None:
        lm_head.bias.requires_grad_(True)

    emb_mask = torch.zeros((token_emb.weight.shape[0], 1), device=token_emb.weight.device, dtype=token_emb.weight.dtype)
    emb_mask[old_vocab_size:] = 1.0

    head_w_mask = torch.zeros((lm_head.weight.shape[0], 1), device=lm_head.weight.device, dtype=lm_head.weight.dtype)
    head_w_mask[old_vocab_size:] = 1.0

    handles.append(token_emb.weight.register_hook(lambda g, m=emb_mask: g * m))
    handles.append(lm_head.weight.register_hook(lambda g, m=head_w_mask: g * m))

    if lm_head.bias is not None:
        head_b_mask = torch.zeros((lm_head.bias.shape[0],), device=lm_head.bias.device, dtype=lm_head.bias.dtype)
        head_b_mask[old_vocab_size:] = 1.0
        handles.append(lm_head.bias.register_hook(lambda g, m=head_b_mask: g * m))

    return handles

def count_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = base_model.to(device)

old_vocab_size_for_delta = model.bert.embedding.token.num_embeddings
if len(vocab) != model.bert.embedding.token.num_embeddings:
    old_vocab_size_for_delta = resize_vocab_dependent_layers(model, len(vocab))

if LORA_CONFIG['freeze_base_model']:
    for p in model.parameters():
        p.requires_grad = False

applied_lora_targets = inject_lora(
    model,
    candidate_lora_targets,
    r=LORA_CONFIG['r'],
    alpha=LORA_CONFIG['alpha'],
    dropout=LORA_CONFIG['dropout'],
)

row_grad_mask_handles = []
if LORA_CONFIG['train_new_vocab_rows'] and len(vocab) > old_vocab_size_for_delta:
    row_grad_mask_handles = enable_new_vocab_row_training(model, old_vocab_size_for_delta)

model = model.to(device)

total_params, trainable_params = count_params(model)
print('Applied LoRA to modules:', len(applied_lora_targets))
for n in applied_lora_targets:
    print('  ', n)
print('Vocab size in model:', model.bert.embedding.token.num_embeddings)
print('Old vocab size:', old_vocab_size_for_delta)
print('Total params:', f'{total_params:,}')
print('Trainable params:', f'{trainable_params:,}')
print('Trainable ratio:', f'{100 * trainable_params / total_params:.4f}%')

print('Trainable tensors:')
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f'  {name}: {tuple(p.shape)}')

## 9) Build Train / Validation DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from training.bert_pytorch.dataset.sample import generate_train_valid, fixed_window
from training.bert_pytorch.dataset.log_dataset import LogDataset


def generate_train_valid_robust(
    data_path,
    window_size=20,
    adaptive_window=True,
    sample_ratio=1.0,
    valid_size=0.1,
    seq_len=None,
    min_len=0,
):
    try:
        return generate_train_valid(
            data_path,
            window_size=window_size,
            adaptive_window=adaptive_window,
            sample_ratio=sample_ratio,
            valid_size=valid_size,
            seq_len=seq_len,
            min_len=min_len,
        )
    except ValueError as exc:
        print('Fallback to robust split because generate_train_valid failed:', exc)

    with open(data_path, 'r', encoding='utf-8') as f:
        data_iter = f.readlines()

    num_session = int(len(data_iter) * sample_ratio)
    num_session = max(1, min(num_session, len(data_iter)))

    test_size = int(num_session * valid_size)
    test_size = max(1, test_size)
    test_size = min(test_size, max(1, num_session - 1))

    logkey_seq_pairs = []
    time_seq_pairs = []

    for i, line in enumerate(tqdm(data_iter, desc='robust_split')):
        if i >= num_session:
            break
        logkeys, times = fixed_window(line, window_size, adaptive_window, seq_len, min_len)
        logkey_seq_pairs.extend(logkeys)
        time_seq_pairs.extend(times)

    if len(logkey_seq_pairs) < 2:
        raise RuntimeError('Not enough sequences to split into train/valid.')

    idx = np.arange(len(logkey_seq_pairs))
    train_idx, valid_idx = train_test_split(idx, test_size=min(test_size, len(idx) - 1), random_state=1234)

    logkey_trainset = [logkey_seq_pairs[i] for i in train_idx]
    logkey_validset = [logkey_seq_pairs[i] for i in valid_idx]
    time_trainset = [time_seq_pairs[i] for i in train_idx]
    time_validset = [time_seq_pairs[i] for i in valid_idx]

    train_sort_index = np.argsort(-1 * np.array(list(map(len, logkey_trainset))))
    valid_sort_index = np.argsort(-1 * np.array(list(map(len, logkey_validset))))

    logkey_trainset = np.array([logkey_trainset[i] for i in train_sort_index], dtype=object)
    logkey_validset = np.array([logkey_validset[i] for i in valid_sort_index], dtype=object)
    time_trainset = np.array([time_trainset[i] for i in train_sort_index], dtype=object)
    time_validset = np.array([time_validset[i] for i in valid_sort_index], dtype=object)

    print('Robust split -> train seqs:', len(logkey_trainset), 'valid seqs:', len(logkey_validset))
    return logkey_trainset, logkey_validset, time_trainset, time_validset


logkey_train, logkey_valid, time_train, time_valid = generate_train_valid_robust(
    str(train_file_path),
    window_size=HYPERPARAMS['window_size'],
    adaptive_window=HYPERPARAMS['adaptive_window'],
    sample_ratio=HYPERPARAMS['train_ratio'],
    valid_size=HYPERPARAMS['valid_ratio'],
    seq_len=HYPERPARAMS['seq_len'],
    min_len=HYPERPARAMS['min_len'],
)

train_dataset = LogDataset(
    logkey_train,
    time_train,
    vocab=vocab,
    seq_len=HYPERPARAMS['seq_len'],
    on_memory=True,
    mask_ratio=HYPERPARAMS['mask_ratio'],
)
valid_dataset = LogDataset(
    logkey_valid,
    time_valid,
    vocab=vocab,
    seq_len=HYPERPARAMS['seq_len'],
    on_memory=True,
    mask_ratio=HYPERPARAMS['mask_ratio'],
)

drop_last_train = len(train_dataset) >= HYPERPARAMS['batch_size']

train_loader = DataLoader(
    train_dataset,
    batch_size=HYPERPARAMS['batch_size'],
    shuffle=True,
    num_workers=HYPERPARAMS['num_workers'],
    collate_fn=train_dataset.collate_fn,
    drop_last=drop_last_train,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=HYPERPARAMS['batch_size'],
    shuffle=False,
    num_workers=HYPERPARAMS['num_workers'],
    collate_fn=valid_dataset.collate_fn,
    drop_last=False,
)

print('Train sequences:', len(train_dataset))
print('Valid sequences:', len(valid_dataset))
print('Train batches:', len(train_loader))
print('Valid batches:', len(valid_loader))



## 10) Fine-tuning Logic (LoRA)

In [ ]:
import json
import math
import time

criterion = nn.NLLLoss(ignore_index=0)

def create_optimizer_and_scheduler(model, train_loader, total_epochs):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError('No trainable parameters found.')

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=HYPERPARAMS['lr'],
        weight_decay=HYPERPARAMS['weight_decay'],
    )

    updates_per_epoch = max(1, math.ceil(len(train_loader) / HYPERPARAMS['grad_accum_steps']))
    total_updates = max(1, updates_per_epoch * total_epochs)
    warmup_updates = int(total_updates * HYPERPARAMS['warmup_ratio'])

    def lr_lambda(current_step):
        if current_step < warmup_updates:
            return float(current_step) / float(max(1, warmup_updates))
        remain = total_updates - current_step
        decay_steps = max(1, total_updates - warmup_updates)
        return max(0.0, float(remain) / float(decay_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler, trainable_params

def run_one_epoch(model, data_loader, optimizer, scheduler, scaler, train_mode, trainable_params):
    model.train(mode=train_mode)

    total_loss = 0.0
    steps = 0

    if train_mode:
        optimizer.zero_grad(set_to_none=True)

    amp_enabled = bool(HYPERPARAMS['use_amp'] and device.type == 'cuda')

    for step_idx, batch in enumerate(data_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            result = model(batch['bert_input'], batch['time_input'])
            logits = result['logkey_output']
            loss = criterion(logits.transpose(1, 2), batch['bert_label'])
            loss = loss / HYPERPARAMS['grad_accum_steps']

        if train_mode:
            scaler.scale(loss).backward()

            should_step = (step_idx % HYPERPARAMS['grad_accum_steps'] == 0) or (step_idx == len(data_loader))
            if should_step:
                if HYPERPARAMS['max_grad_norm'] is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable_params, HYPERPARAMS['max_grad_norm'])

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        total_loss += loss.item() * HYPERPARAMS['grad_accum_steps']
        steps += 1

    return total_loss / max(1, steps)

def extract_lora_state_dict(model):
    state = {}
    for k, v in model.state_dict().items():
        if ('.lora_A' in k) or ('.lora_B' in k):
            state[k] = v.detach().cpu()
    return state

def extract_vocab_delta(model, old_vocab_size):
    current_vocab_size = model.bert.embedding.token.num_embeddings
    if current_vocab_size <= old_vocab_size:
        return {}

    delta = {
        'token_embedding_rows': model.bert.embedding.token.weight.detach().cpu()[old_vocab_size:].clone(),
        'mask_lm_weight_rows': model.mask_lm.linear.weight.detach().cpu()[old_vocab_size:].clone(),
    }
    if model.mask_lm.linear.bias is not None:
        delta['mask_lm_bias_rows'] = model.mask_lm.linear.bias.detach().cpu()[old_vocab_size:].clone()
    return delta

def save_checkpoint(path, epoch, model, optimizer, scheduler, scaler, best_val_loss, history):
    payload = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'history': history,
        'paths': PATHS,
        'hyperparams': HYPERPARAMS,
        'lora_config': LORA_CONFIG,
        'vocab_path': str(vocab_path_for_training),
        'old_vocab_size_for_delta': int(old_vocab_size_for_delta),
    }
    torch.save(payload, path)

def train_loop(model, train_loader, valid_loader, num_epochs, resume_state=None):
    ckpt_dir = output_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    optimizer, scheduler, trainable_params = create_optimizer_and_scheduler(model, train_loader, total_epochs=num_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=bool(HYPERPARAMS['use_amp'] and device.type == 'cuda'))

    start_epoch = 0
    best_val_loss = float('inf')
    history = []

    if resume_state is not None:
        model.load_state_dict(resume_state['model_state'], strict=True)
        optimizer.load_state_dict(resume_state['optimizer_state'])
        scheduler.load_state_dict(resume_state['scheduler_state'])
        if resume_state.get('scaler_state') is not None:
            scaler.load_state_dict(resume_state['scaler_state'])

        start_epoch = int(resume_state['epoch']) + 1
        best_val_loss = float(resume_state.get('best_val_loss', best_val_loss))
        history = list(resume_state.get('history', []))
        print(f'Resuming from epoch {start_epoch}')

    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()
        train_loss = run_one_epoch(model, train_loader, optimizer, scheduler, scaler, train_mode=True, trainable_params=trainable_params)
        val_loss = run_one_epoch(model, valid_loader, optimizer, scheduler, scaler, train_mode=False, trainable_params=trainable_params)

        row = {
            'epoch': epoch,
            'train_loss': float(train_loss),
            'val_loss': float(val_loss),
            'lr': float(optimizer.param_groups[0]['lr']),
            'seconds': float(time.time() - t0),
        }
        history.append(row)

        print(f"Epoch {epoch}: train_loss={train_loss:.6f} val_loss={val_loss:.6f} lr={row['lr']:.6e} time={row['seconds']:.1f}s")

        epoch_ckpt = ckpt_dir / f'epoch_{epoch:03d}.pt'
        save_checkpoint(epoch_ckpt, epoch, model, optimizer, scheduler, scaler, best_val_loss, history)

        if val_loss < best_val_loss:
            best_val_loss = float(val_loss)
            best_ckpt = ckpt_dir / 'best.pt'
            save_checkpoint(best_ckpt, epoch, model, optimizer, scheduler, scaler, best_val_loss, history)
            print('  -> best checkpoint updated:', best_ckpt)

        history_path = output_dir / 'train_history.csv'
        pd.DataFrame(history).to_csv(history_path, index=False)

    return {
        'history': history,
        'best_val_loss': best_val_loss,
        'best_checkpoint': str((output_dir / 'checkpoints' / 'best.pt').resolve()),
    }

## 11) Launch Training (Required)

In [ ]:
RUN_TRAINING = True
RESUME_FROM_CHECKPOINT = None  # Например: output_dir / 'checkpoints' / 'best.pt'

resume_state = None
if RESUME_FROM_CHECKPOINT is not None:
    resume_state = torch.load(RESUME_FROM_CHECKPOINT, map_location=device)

if RUN_TRAINING:
    training_state = train_loop(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        num_epochs=HYPERPARAMS['epochs'],
        resume_state=resume_state,
    )
    display(pd.DataFrame(training_state['history']).tail(10))
    print('Best checkpoint:', training_state['best_checkpoint'])
else:
    training_state = {'history': [], 'best_checkpoint': str((output_dir / 'checkpoints' / 'best.pt').resolve())}
    print('Training skipped (RUN_TRAINING=False)')

## 12) Save LoRA Adapters, Checkpoint Metadata, Final Config (Required)

In [ ]:
from datetime import datetime, timezone

artifacts_dir = output_dir / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

adapter_payload = {
    'base_checkpoint': str(base_checkpoint_path),
    'base_model_meta': base_model_meta,
    'lora_config': LORA_CONFIG,
    'lora_state_dict': extract_lora_state_dict(model),
    'vocab_path': str(vocab_path_for_training),
    'old_vocab_size_for_delta': int(old_vocab_size_for_delta),
    'vocab_delta': extract_vocab_delta(model, old_vocab_size_for_delta),
}

lora_adapter_path = artifacts_dir / 'lora_adapter.pt'
torch.save(adapter_payload, lora_adapter_path)

final_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'paths': PATHS,
    'hyperparams': HYPERPARAMS,
    'data_prep': DATA_PREP,
    'lora_config': LORA_CONFIG,
    'base_model_meta': base_model_meta,
    'best_checkpoint': training_state.get('best_checkpoint'),
    'history_rows': len(training_state.get('history', [])),
}

final_config_path = artifacts_dir / 'final_training_config.json'
with final_config_path.open('w', encoding='utf-8') as f:
    json.dump(final_config, f, ensure_ascii=False, indent=2)

history_json_path = artifacts_dir / 'train_history.json'
with history_json_path.open('w', encoding='utf-8') as f:
    json.dump(training_state.get('history', []), f, ensure_ascii=False, indent=2)

print('Saved LoRA adapter:', lora_adapter_path)
print('Saved final config:', final_config_path)
print('Saved history JSON:', history_json_path)

## 13) Resume Training Cell (Required)

In [ ]:
def resume_training_from_checkpoint(checkpoint_path: Path, additional_epochs: int):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    current_epoch = int(checkpoint['epoch'])
    target_total_epochs = current_epoch + 1 + int(additional_epochs)

    print('Resume checkpoint:', checkpoint_path)
    print('Current epoch in checkpoint:', current_epoch)
    print('Will train until epoch:', target_total_epochs - 1)

    return train_loop(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        num_epochs=target_total_epochs,
        resume_state=checkpoint,
    )

RUN_RESUME = False
RESUME_CHECKPOINT = output_dir / 'checkpoints' / 'best.pt'
ADDITIONAL_EPOCHS = 2

if RUN_RESUME:
    resumed_training_state = resume_training_from_checkpoint(RESUME_CHECKPOINT, ADDITIONAL_EPOCHS)
    display(pd.DataFrame(resumed_training_state['history']).tail(10))
else:
    print('Resume skipped (RUN_RESUME=False). Set RUN_RESUME=True to continue training.')

## 14) Quick Smoke Test / Inference (Required)

In [ ]:
SMOKE_SAMPLES = 8
TOP_K = 10

smoke_count = min(SMOKE_SAMPLES, len(logkey_valid))
if smoke_count == 0:
    raise RuntimeError('No validation samples available for smoke test.')

smoke_dataset = LogDataset(
    logkey_valid[:smoke_count],
    time_valid[:smoke_count],
    vocab=vocab,
    seq_len=HYPERPARAMS['seq_len'],
    on_memory=True,
    predict_mode=True,
    mask_ratio=HYPERPARAMS['mask_ratio'],
)

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=smoke_count,
    shuffle=False,
    num_workers=0,
    collate_fn=smoke_dataset.collate_fn,
)

batch = next(iter(smoke_loader))
batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(batch['bert_input'], batch['time_input'])
    logits = out['logkey_output']
    topk_idx = torch.topk(logits, k=TOP_K, dim=-1).indices

rows = []
for i in range(batch['bert_input'].shape[0]):
    mask_positions = torch.where(batch['bert_label'][i] > 0)[0].tolist()
    if not mask_positions:
        continue

    hits = 0
    examples = []
    for pos in mask_positions[:3]:
        true_id = int(batch['bert_label'][i, pos].item())
        pred_ids = topk_idx[i, pos].detach().cpu().tolist()
        hit = int(true_id in pred_ids)
        hits += hit
        examples.append({
            'pos': int(pos),
            'true_token': vocab.itos[true_id] if true_id < len(vocab.itos) else f'<{true_id}>',
            'top1_token': vocab.itos[pred_ids[0]] if pred_ids[0] < len(vocab.itos) else f'<{pred_ids[0]}>',
            'hit@k': hit,
        })

    rows.append({
        'sample_id': i,
        'masked_tokens': len(mask_positions),
        'hits_at_k': hits,
        'hit_rate_at_k': hits / max(1, len(mask_positions)),
        'examples': examples,
    })

smoke_df = pd.DataFrame(rows)
display(smoke_df)
if len(smoke_df) > 0:
    print('Mean hit_rate@k:', round(float(smoke_df['hit_rate_at_k'].mean()), 6))

## 15) Precision / Recall / F1 (from prediction JSONs)

Оценка качества по готовым prediction-файлам: normal-логи vs attack-логи.
Метрики считаются двумя способами: по `is_anomaly` и по лучшему порогу `anomaly_score`.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
import json

PRED_METRICS_CONFIG = {
    'normal_predictions_path': str(repo_root / 'data' / 'loguru_predictions.json'),
    'anomaly_predictions_path': str(repo_root / 'data' / 'loguru_sqli_predictions.json'),
    'normal_label': 0,
    'anomaly_label': 1,
}


def _load_predictions(path):
    with Path(path).open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise TypeError(f'Expected list in {path}, got {type(data)}')
    if len(data) == 0:
        raise ValueError(f'Empty predictions file: {path}')
    return data


normal_preds = _load_predictions(PRED_METRICS_CONFIG['normal_predictions_path'])
anomaly_preds = _load_predictions(PRED_METRICS_CONFIG['anomaly_predictions_path'])

all_preds = normal_preds + anomaly_preds
y_true = np.array(
    [PRED_METRICS_CONFIG['normal_label']] * len(normal_preds) +
    [PRED_METRICS_CONFIG['anomaly_label']] * len(anomaly_preds)
)

# 1) Metrics by boolean is_anomaly
y_pred_flag = np.array([1 if bool(item.get('is_anomaly', False)) else 0 for item in all_preds])

p_flag = precision_score(y_true, y_pred_flag, zero_division=0)
r_flag = recall_score(y_true, y_pred_flag, zero_division=0)
f1_flag = f1_score(y_true, y_pred_flag, zero_division=0)
cm_flag = confusion_matrix(y_true, y_pred_flag, labels=[0, 1])

print('=== Metrics by is_anomaly ===')
print(f'precision: {p_flag:.6f}')
print(f'recall:    {r_flag:.6f}')
print(f'f1:        {f1_flag:.6f}')
print('confusion_matrix [[TN, FP], [FN, TP]]:')
print(cm_flag)

# 2) Best threshold by anomaly_score sweep
scores = np.array([float(item['anomaly_score']) for item in all_preds])
unique_thresholds = np.unique(scores)

best = None  # (f1, precision, recall, threshold, cm)
for thr in unique_thresholds:
    y_pred_thr = (scores >= thr).astype(int)
    p = precision_score(y_true, y_pred_thr, zero_division=0)
    r = recall_score(y_true, y_pred_thr, zero_division=0)
    f1 = f1_score(y_true, y_pred_thr, zero_division=0)
    cm = confusion_matrix(y_true, y_pred_thr, labels=[0, 1])

    if best is None or f1 > best[0]:
        best = (f1, p, r, float(thr), cm)

print('\n=== Best metrics by anomaly_score threshold sweep ===')
print(f'threshold: {best[3]:.18f}')
print(f'precision: {best[1]:.6f}')
print(f'recall:    {best[2]:.6f}')
print(f'f1:        {best[0]:.6f}')
print('confusion_matrix [[TN, FP], [FN, TP]]:')
print(best[4])

metrics_summary = {
    'by_is_anomaly': {
        'precision': float(p_flag),
        'recall': float(r_flag),
        'f1': float(f1_flag),
        'confusion_matrix': cm_flag.tolist(),
    },
    'by_best_threshold': {
        'threshold': float(best[3]),
        'precision': float(best[1]),
        'recall': float(best[2]),
        'f1': float(best[0]),
        'confusion_matrix': best[4].tolist(),
    },
    'counts': {
        'normal_samples': int(len(normal_preds)),
        'anomaly_samples': int(len(anomaly_preds)),
        'total': int(len(all_preds)),
    },
}

metrics_out = output_dir / 'artifacts' / 'metrics_summary.json'
metrics_out.parent.mkdir(parents=True, exist_ok=True)
with metrics_out.open('w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

print('\nSaved metrics to:', metrics_out)

